In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
pd.options.display.max_colwidth = None

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test
INPUT_CSV = "inputs/bsa.csv"
BASE_OUTPUT_FOLDER = "outputs_notebook"

METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

RUN_NAME = Path(INPUT_CSV).stem
REFERENCE_MODE = True
CHAIN = ""

CONFIDENCE_THRESHOLD = 0.8
MIN_LENGTH = 7
FDR_THRESHOLD = 0.1

#MASS_ERR_LIMIT = 20
#MAX_IRT_ERROR = 60
#MIN_ENTROPY = 1
#PROSIT_FILTER = True
#Z_SCORE_THRESHOLD = -0.5


# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
KMER_SIZE = 7
MIN_OVERLAP = 3
SIZE_THRESHOLD = 10
MIN_IDENTITY = 0.8
MAX_MISMATCHES = 100

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
experiment_folder = base_output_folder / run_folder_name


run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")
logger.info(f"All results will be saved to: {experiment_folder}")

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

In [ ]:
original_data = pd.read_csv(INPUT_CSV)

In [ ]:
original_data

In [ ]:
data = original_data.copy()

In [ ]:
data.columns

In [ ]:
data.head(2)

In [ ]:
data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

In [ ]:
data["protease"] = data["spectrum_id"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

#protease_col = data.pop("protease")
#data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

In [ ]:
data.columns

In [ ]:
data

In [ ]:
data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

# move cleaned_preds next to prediction_untokenised
#cleaned_preds_col = data.pop("cleaned_preds")
#data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

In [ ]:
cleaned_psms = data["cleaned_preds"].tolist()

In [ ]:
filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [ ]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [ ]:
data.head(3)

In [ ]:
#data.drop(columns=['prediction_untokenised'], inplace=True)

In [ ]:
data.loc[:, "mapped"] = data["cleaned_preds"].apply(lambda x: x in protein_norm)

In [ ]:
data = data[data['cleaned_preds'].str.len() >= MIN_LENGTH]

In [ ]:
MAX_LENGTH = 20

In [ ]:
# remocve sequences longer than MAX_LENGTH
data = data[data['cleaned_preds'].str.len() <= MAX_LENGTH]

In [ ]:
# show me value counts of mapped vs unmapped
data['mapped'].value_counts()

In [ ]:
# remove the column "instanovo_token_log_probabilities"
data = data.drop(columns=["instanovo_token_log_probabilities"])

In [ ]:
data.head()

## Distribution

In [ ]:
import matplotlib.patches as mpatches

def plot_map_unmap_distribution(df, run, folder, conf_lim, ratio=1, unique_peps=True, title=False):
    visualization.set_publication_style()
    
    df_filtered = df[df["conf"] >= conf_lim].copy()
    if unique_peps:
        df_filtered = df_filtered.sort_values("conf", ascending=False).drop_duplicates(subset=["cleaned_preds"])

    _, ax = plt.subplots(figsize=visualization.get_figsize(width_ratio=ratio))
    
    c_maps, c_not = "#4A90E2", "#FF9F1C"
    bins = np.arange(0, 1.01, 0.01)

    sns.histplot(
        data=df_filtered, x="conf", hue="mapped",
        palette={True: c_maps, False: c_not},
        bins=bins, element="bars", fill=True, alpha=0.8,
        multiple="layer", ax=ax, legend=False
    )

    overlap_color = "#8c7c6d"
    handles = [
        mpatches.Patch(color=c_maps, alpha=0.8, label='Maps in protein'),
        mpatches.Patch(color=c_not, alpha=0.8, label='Not in protein'),
        mpatches.Patch(color=overlap_color, alpha=0.8, label='Overlap')
    ]
    ax.legend(
        handles=handles, loc='lower center', 
        bbox_to_anchor=(0.5, 1.02), ncol=3, borderaxespad=0.,
        frameon=False
    )

    ax.set_xlim(0, 1)
    ax.set_yscale("log")
    ax.set_xlabel("Confidence (Calibrated)")
    ax.set_ylabel("Unique Peptides" if unique_peps else "PSMs counts")
    
    if title:
        ax.set_title("Sequence Distribution by Confidence", pad=30)

    sns.despine()

    Path(folder).mkdir(parents=True, exist_ok=True)
    suffix = "unique" if unique_peps else "all"
    output_path = f"{folder}/fig2a_{run}_distribution_psm_{suffix}.svg"
    
    plt.savefig(output_path, format="svg", bbox_inches="tight")
    plt.close()

    return output_path

In [ ]:
plot_map_unmap_distribution(data, RUN_NAME, FIGURES_DIR, 0, 1, False, title=False)

In [ ]:
def plot_dual_quality_distributions(run, df, reference, folder, ratio=1):
    """
    Plots confidence and FDR distributions using a square ratio (1:1).
    """
    visualization.set_publication_style()
    
    df_processed = df.copy()
    df_processed["mapped_status"] = df_processed["cleaned_preds"].apply(
        lambda x: "Maps to protein" if isinstance(x, str) and x in reference else "Not in protein"
    )
    
    palette = {
        "Maps to protein": "#4A90E2",
        "Not in protein": "#FF9F1C"
    }
    hue_order = ["Not in protein", "Maps to protein"]

    df_conf = df_processed[df_processed["conf"] >= 0.90].copy()
    
    fig1, ax1 = plt.subplots(figsize=visualization.get_figsize(width_ratio=ratio))
    
    sns.histplot(
        data=df_conf,
        x="conf",
        hue="mapped_status",
        multiple="stack",
        palette=palette,
        hue_order=hue_order,
        bins=np.arange(0.90, 1.01, 0.02),
        shrink=0.8,
        edgecolor="none",
        alpha=0.8,
        ax=ax1,
        legend=False
    )

    ax1.set_xlabel("Confidence (calibrated)")
    ax1.set_ylabel("PSMs counts")
    ax1.set_xlim(0.9, 1.0)
    
    ax1.legend(
        handles=[mpatches.Patch(color=palette[label], label=label) for label in hue_order],
        loc='lower center', 
        bbox_to_anchor=(0.5, 1.02), 
        ncol=2,
        frameon=False
    )
    
    sns.despine()
    
    if folder:
        Path(folder).mkdir(parents=True, exist_ok=True)
        plt.savefig(f"{folder}/fig2b_{run}_confidence_barplot.svg", format="svg", bbox_inches="tight")
    plt.show()

    df_fdr = df_processed[df_processed["psm_q_value"] <= 0.20].copy()
    
    fdr_bins = [0, 0.01, 0.05, 0.10, 0.20]
    fdr_labels = ["0-1%", "1-5%", "5-10%", "10-20%"]
    df_fdr["fdr_interval"] = pd.cut(df_fdr["psm_q_value"], bins=fdr_bins, labels=fdr_labels, include_lowest=True)

    fig2, ax2 = plt.subplots(figsize=visualization.get_figsize(width_ratio=ratio))
    
    sns.histplot(
        data=df_fdr,
        x="fdr_interval",
        hue="mapped_status",
        multiple="stack",
        palette=palette,
        hue_order=hue_order,
        shrink=0.7,
        edgecolor="none",
        alpha=0.8,
        discrete=True,
        ax=ax2,
        legend=False
    )

    ax2.set_xlabel("FDR interval (q-value)")
    ax2.set_ylabel("PSMs counts")
    
    ax2.legend(
        handles=[mpatches.Patch(color=palette[label], label=label) for label in hue_order],
        loc='lower center', 
        bbox_to_anchor=(0.5, 1.02), 
        ncol=2,
        frameon=False
    )

    sns.despine()

    if folder:
        plt.savefig(f"{folder}/fig2e_{run}_fdr_barplots.svg", format="svg", bbox_inches="tight")
    plt.show()

In [ ]:
plot_dual_quality_distributions(RUN_NAME, data, reference=protein_norm, folder=FIGURES_DIR, ratio=1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os
from pathlib import Path

def plot_fdr_mapped_unmapped_ratio(run, df, reference, folder, ratio=1, unique_peps=False):
    """
    Plots the Mapped/Unmapped ratio across the full FDR range (0 to 1).
    """
    visualization.set_publication_style()
    
    df_copy = df.copy()
    
    df_copy["mapped_status"] = df_copy["cleaned_preds"].apply(
        lambda x: "mapped" if isinstance(x, str) and x in reference else "unmapped"
    )
    
    if unique_peps:
        df_copy = df_copy.sort_values("conf", ascending=False).drop_duplicates(subset=["cleaned_preds"])
    
    # Estendiamo i thresholds fino a 1.0
    thresholds = np.linspace(0.01, 1.0, 50)
    data_points = []

    for t in thresholds:
        subset = df_copy[df_copy["psm_q_value"] <= t]
        if len(subset) == 0: continue
            
        counts = subset["mapped_status"].value_counts()
        n_mapped = counts.get("mapped", 0)
        n_unmapped = counts.get("unmapped", 0)
        
        ratio_val = n_mapped / n_unmapped if n_unmapped > 0 else np.nan
        data_points.append({"fdr": t, "ratio": ratio_val})

    plot_df = pd.DataFrame(data_points).dropna()

    fig, ax = plt.subplots(figsize=visualization.get_figsize(width_ratio=ratio))

    VIBRANT_BLUE = "#4A90E2"
    VIBRANT_ORANGE = "#FF9F1C"
    
    sns.lineplot(
        data=plot_df, x='fdr', y='ratio', 
        color=VIBRANT_BLUE, linewidth=1.5, ax=ax, zorder=2
    )

    # Manteniamo i target thresholds classici per i punti di interesse
    target_thresholds = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
    highlights = []
    for target in target_thresholds:
        idx = (np.abs(plot_df['fdr'] - target)).argmin()
        highlights.append(plot_df.iloc[idx])
    
    highlights_df = pd.DataFrame(highlights)
    
    ax.scatter(
        highlights_df['fdr'], highlights_df['ratio'], 
        s=70, facecolors='white', edgecolors=VIBRANT_ORANGE, 
        linewidth=1.5, zorder=3
    )

    ax.set_xlabel("FDR Threshold (q-value)")
    ax.set_ylabel("Ratio (Mapped / Unmapped)")
    
    # Impostiamo i limiti e i ticks dell'asse X da 0 a 1
    ax.set_xlim(0, 1.0)
    ax.set_xticks(np.arange(0, 1.1, 0.1))
    
    ax.grid(True, which='major', axis='both', color='#f0f0f0', linestyle='-', linewidth=0.5)
    
    sns.despine()

    if folder:
        Path(folder).mkdir(parents=True, exist_ok=True)
        out_name = f"fig2c_{run}_ratio_full_range.svg"
        plt.savefig(os.path.join(folder, out_name), format="svg", bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_fdr_mapped_unmapped_ratio(RUN_NAME, data, protein_norm, FIGURES_DIR)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from pathlib import Path

def plot_coverage_vs_fdr_curve(run, df, reference_protein, folder, min_identity=0.8, chain="Light", ratio=1):
    visualization.set_publication_style()
    
    df_copy = df.copy()
    fdr_thresholds = np.linspace(0, 1.0, 50) # Aumentato a 50 per una curva fluidissima
    coverage_data = []

    temp_stats_folder = os.path.join(folder, "temp_stats_calc")
    os.makedirs(temp_stats_folder, exist_ok=True)

    for fdr in tqdm(fdr_thresholds):
        subset = df_copy[df_copy["psm_q_value"] <= fdr]
        if subset.empty:
            coverage_data.append({"fdr": fdr, "coverage": 0.0})
            continue

        sequences = subset["cleaned_preds"].tolist()
        try:
            mapped_psms = visualization.process_protein_contigs_scaffold(
                sequences, reference_protein, max_mismatches=10, min_identity=min_identity
            )
            df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_psms)
            stats = helpers.compute_assembly_statistics(
                df_mapped, f"temp_{chain}_{fdr:.2f}", temp_stats_folder, reference_protein
            )
            
            cov_value = 0
            for k in ["Coverage", "Protein Coverage", "coverage", "protein_coverage"]:
                if k in stats:
                    val = stats[k]
                    cov_value = float(val.replace("%", "")) if isinstance(val, str) else float(val)
                    if cov_value <= 1.0 and cov_value > 0:
                        cov_value *= 100
                    break
            coverage_data.append({"fdr": fdr, "coverage": cov_value})
        except:
            coverage_data.append({"fdr": fdr, "coverage": 0.0})

    plot_df = pd.DataFrame(coverage_data).sort_values("fdr") # Ordine garantito

    fig, ax = plt.subplots(figsize=visualization.get_figsize(width_ratio=ratio))
    VIBRANT_BLUE = "#4A90E2"

    # Uso plt.plot per controllo totale: '-' forza la linea continua
    ax.plot(
        plot_df["fdr"], 
        plot_df["coverage"], 
        color=VIBRANT_BLUE, 
        linestyle='-', 
        linewidth=2, 
        zorder=2
    )

    # Aggiungo marker solo ai punti critici per non appesantire la linea
    critical_fdr = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
    for val in critical_fdr:
        # Trova il punto più vicino
        idx = (plot_df['fdr'] - val).abs().idxmin()
        ax.scatter(
            plot_df.loc[idx, 'fdr'], 
            plot_df.loc[idx, 'coverage'], 
            color='white', 
            edgecolor=VIBRANT_BLUE, 
            s=40, 
            zorder=3, 
            linewidth=1.2
        )

    ax.set_xlabel("FDR Threshold (q-value)", fontweight='normal')
    ax.set_ylabel("Protein Coverage (%)", fontweight='normal')
    ax.set_xlim(0, 1.0)
    ax.set_ylim(0, 105)
    ax.set_xticks(np.arange(0, 1.1, 0.2))
    
    ax.grid(True, which='major', axis='both', color='#f0f0f0', linestyle='-', linewidth=0.5)
    sns.despine()

    if folder:
        Path(folder).mkdir(parents=True, exist_ok=True)
        plt.savefig(os.path.join(folder, f"fig2d_{run}_{chain}_coverage_solid.svg"), format="svg", bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_coverage_vs_fdr_curve(RUN_NAME, data, protein_norm, FIGURES_DIR)

In [ ]:
def plot_protease_confidence_ridges(df, colors_path='json/protease_colors.json', folder=None):
    visualization.set_publication_style()
    
    with open(colors_path, 'r') as f:
        protease_colors = json.load(f)

    proteases = sorted(df['protease'].unique())
    n_prot = len(proteases)

    total_w, total_h = visualization.get_figsize(width_ratio=1)
    row_height = total_h / n_prot

    sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
    
    g = sns.FacetGrid(
        df, row="protease", hue="protease", 
        aspect=1, 
        height=row_height, 
        palette=protease_colors,
        row_order=proteases
    )

    g.map(sns.kdeplot, "conf", log_scale=True, bw_adjust=.7, fill=True, alpha=0.8, linewidth=0)
    
    g.map(plt.axhline, y=0, lw=1, clip_on=False, color='#333333', alpha=0.8)

    def label(x, color, label):
        ax = plt.gca()
        ax.text(1.02, .1, label, fontweight="normal", color="black",
                ha="left", va="center", transform=ax.transAxes, fontsize=12)

    g.map(label, "conf")

    g.set_titles("")
    g.set(yticks=[], ylabel="")
    g.despine(bottom=True, left=True)
    
    g.figure.subplots_adjust(hspace=0.4) 
    g.figure.set_size_inches(total_w, total_h)

    tick_vals = [1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1] 
    tick_labels = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]

    for ax in g.axes.flat:
        ax.set_xlim(1e-10, 1)
        ax.set_xticks(tick_vals)
        ax.set_xticklabels(tick_labels)
        for label in ax.get_xticklabels():
            label.set_fontweight('normal')

    plt.xlabel("Confidence", fontsize=15, fontweight='normal', labelpad=20)
    
    if folder:
        Path(folder).mkdir(parents=True, exist_ok=True)
        g.savefig(f"{folder}/supp_fig1a_ridges_proteases.svg", format="svg", bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_protease_confidence_ridges(data, colors_path='json/protease_colors.json', folder=FIGURES_DIR)

## Add quantification data

In [ ]:
def add_quantification_data(df_main, run_name, fdr_threshold, inputs_folder="inputs"):
    """
    Filters df_main by FDR, then looks for a quantification file ({run_name}_quant_scores.csv).
    Merges the abundance data into the filtered dataframe.
    """
    if fdr_threshold is not None:
        if "psm_q_value" in df_main.columns:
            initial_len = len(df_main)
            df_main = df_main[df_main['psm_q_value'] <= fdr_threshold].copy()
            logger.info(f"FDR Filter applied inside merge function: {initial_len} -> {len(df_main)} rows (<= {fdr_threshold})")
        else:
            logger.warning("FDR threshold provided but 'psm_q_value' column missing. Skipping filter.")

    quant_file_name = f"{run_name}_quant_scores.csv"
    quant_file_path = Path(inputs_folder) / quant_file_name
    
    if not quant_file_path.exists():
        logger.warning(f"Quantification file NOT FOUND: {quant_file_path}")
        logger.warning("Skipping abundance merging. 'peptide_abundance' will be missing.")
        return df_main

    logger.info(f"Found quantification file: {quant_file_path}")
    
    try:
        df_quant = pd.read_csv(quant_file_path)
        
        if "cleaned_preds" not in df_quant.columns or "total_abundance_norm" not in df_quant.columns:
            logger.warning(f"Quantification file format error. Missing columns in {quant_file_path}")
            return df_main

        df_quant_summed = df_quant.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()  
        df_quant_summed.rename(columns={'total_abundance_norm': 'peptide_abundance'}, inplace=True) 
        df_merged = pd.merge(df_main, df_quant_summed, on='cleaned_preds', how='left')
        df_merged['peptide_abundance'] = df_merged['peptide_abundance'].fillna(0)
        
        logger.info(f"Quantification data merged successfully. Output rows: {len(df_merged)}")
        return df_merged

    except Exception as e:
        logger.error(f"Error merging quantification data: {e}")
        return df_main

In [ ]:
print(FDR_THRESHOLD)

In [ ]:
data_abundance = add_quantification_data(data, RUN_NAME, FDR_THRESHOLD)

## Pie chart sunberst

In [ ]:
def plot_sunburst(df, output_folder, output_file, json_colors_path):
    visualization.set_publication_style()
    
    protease_colors_map = {}
    if os.path.exists(json_colors_path):
        with open(json_colors_path, 'r') as f:
            protease_colors_map = json.load(f)
    
    fallback_palette = ['#8dd3c7','#ffffb3','#bebada','#fb8072','#80b1d3','#fdb462','#b3de69','#fccde5']

    df = df.copy()
    df['is_mapped'] = df['mapped'].astype(str).str.lower().isin(['true', 'mapped', 'yes', '1'])

    counts = df.groupby(['protease', 'is_mapped']).size().unstack(fill_value=0).reset_index()
    counts.columns = [str(c) for c in counts.columns]
    
    if 'True' not in counts.columns: counts['True'] = 0
    if 'False' not in counts.columns: counts['False'] = 0
    counts.rename(columns={'True': 'mapped_count', 'False': 'unmapped_count'}, inplace=True)
    
    counts['total'] = counts['mapped_count'] + counts['unmapped_count']
    total_dataset_psms = counts['total'].sum()
    counts = counts.sort_values('total', ascending=False).reset_index(drop=True)

    colors_list = []
    for idx, row in counts.iterrows():
        colors_list.append(protease_colors_map.get(row['protease'], fallback_palette[idx % len(fallback_palette)]))
    counts['color'] = colors_list

    inner_sizes = counts['total'].values
    inner_colors = counts['color'].values
    inner_labels = [f"{row['total']/total_dataset_psms*100:.1f}%" for _, row in counts.iterrows()]
    
    outer_sizes, outer_colors, outer_labels = [], [], []
    for _, row in counts.iterrows():
        mapped, unmapped, total, c = row['mapped_count'], row['unmapped_count'], row['total'], row['color']
        outer_sizes.extend([mapped, unmapped])
        outer_colors.extend([c, (0,0,0,0)])
        eff_pct = (mapped / total * 100) if total > 0 else 0
        outer_labels.extend([f"{eff_pct:.0f}%" if (mapped/total_dataset_psms > 0.01) else "", ""])

    fig, ax = plt.subplots(figsize=visualization.get_figsize(width_ratio=2))
    
    wedges_inner, texts_inner = ax.pie(
        inner_sizes, radius=0.7, colors=inner_colors, labels=inner_labels,
        labeldistance=0.5, startangle=90, counterclock=False, 
        wedgeprops=dict(width=0.4, edgecolor='white', linewidth=1.5) 
    )
    
    for t in texts_inner:
        t.set_size(9)
        t.set_fontweight("normal")

    wedges_outer, texts_outer = ax.pie(
        outer_sizes, radius=1.0, colors=outer_colors, labels=outer_labels,
        labeldistance=0.85, startangle=90, counterclock=False,
        wedgeprops=dict(width=0.25, edgecolor='white', linewidth=1)
    )

    for i, wedge in enumerate(wedges_outer):
        if i % 2 != 0: wedge.set_edgecolor('none')

    for t in texts_outer:
        t.set_size(9)
        t.set_fontweight("normal")

    ax.legend(
        wedges_inner, counts['protease'],
        title="Proteases",
        loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1),
        frameon=False
    )

    ax.add_artist(plt.Circle((0,0), 0.3, fc='white'))
    
    if output_folder and output_file:
        Path(output_folder).mkdir(parents=True, exist_ok=True)
        plt.savefig(os.path.join(output_folder, output_file), format='svg', bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_sunburst(data_abundance, FIGURES_DIR, f"fig2f_{RUN_NAME}_sunburst.svg", "json/protease_colors.json")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
import json
import os
from tqdm import tqdm
from pathlib import Path

def mapping_psms_protease_associated_seaborn(
    mapped_sequences,
    prot_seq,
    labels,
    title,
    output_folder,
    output_file,
    json_colors_path=None,
    show_figure=False,
):
    visualization.set_publication_style()
    
    protease_colors_map = {}
    if json_colors_path and os.path.exists(json_colors_path):
        with open(json_colors_path, 'r') as f:
            protease_colors_map = json.load(f)
    
    unique_labels = sorted(list(set(labels)))
    fallback_palette = sns.color_palette("husl", len(unique_labels)).as_hex()
    label_color = {
        lab: protease_colors_map.get(lab, fallback_palette[i % len(fallback_palette)]) 
        for i, lab in enumerate(unique_labels)
    }
    
    fig_w, fig_h = visualization.get_figsize(width_ratio=3.3)
    # Rimuoviamo il bordo della figura stessa impostando linewidth=0
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), linewidth=0)

    # Fondamentale: linewidth=0 sul rettangolo di sfondo
    ax.add_patch(patches.Rectangle(
        (0, 0), len(prot_seq), 1.0, 
        linewidth=0, 
        facecolor='#f7f7f7', 
        zorder=0,
        edgecolor='none'
    ))

    tracks = {}
    bar_height = 0.6
    track_spacing = 1.0
    base_y_offset = 1.5

    for idx, (_, mapping) in tqdm(enumerate(mapped_sequences), desc="Plotting peptides", total=len(mapped_sequences)):
        start_index, end_index, _, _ = mapping
        lab = labels[idx]
        placed = False
        
        for track_num in sorted(tracks.keys()):
            if not any(max(s, start_index) < min(e, end_index) for s, e in tracks[track_num]):
                tracks[track_num].append((start_index, end_index))
                current_y = base_y_offset + (track_num * track_spacing)
                placed = True
                break
        
        if not placed:
            new_track_num = len(tracks)
            tracks[new_track_num] = [(start_index, end_index)]
            current_y = base_y_offset + (new_track_num * track_spacing)

        # linewidth=0 sui rettangoli dei peptidi
        ax.add_patch(patches.Rectangle(
            (start_index, current_y), end_index - start_index, bar_height,
            linewidth=0, facecolor=label_color[lab], alpha=0.85, edgecolor='none'
        ))

    max_track = len(tracks) if tracks else 0
    ax.set_xlim(0, len(prot_seq))
    ax.set_ylim(0, base_y_offset + (max_track * track_spacing) + 1)
    
    ax.set_title(title, fontweight='normal', pad=30)
    ax.set_xlabel("Residue Position", fontweight='normal')
    ax.set_yticks([])
    
    # Pulizia radicale degli assi
    sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)
    ax.set_axis_off() # Se non ti servono i numeri dell'asse X, questo elimina tutto
    # Se ti servono i numeri X, commenta ax.set_axis_off() e usa:
    # ax.xaxis.set_visible(True)
    # ax.spines['bottom'].set_visible(False)

    legend_patches = [patches.Patch(color=label_color[lab], label=lab, linewidth=0) for lab in unique_labels]
    
    leg = ax.legend(
        handles=legend_patches, 
        title="Proteases",
        loc='upper center', 
        bbox_to_anchor=(0.5, 1.08),
        ncol=min(len(unique_labels), 5),
        frameon=False,
        fontsize=11
    )
    plt.setp(leg.get_title(), fontweight='normal')

    if output_folder and output_file:
        Path(output_folder).mkdir(parents=True, exist_ok=True)
        # Salvataggio senza bordi esterni
        plt.savefig(
            os.path.join(output_folder, output_file), 
            format='svg', 
            bbox_inches='tight', 
            pad_inches=0,
            transparent=True
        )

    if show_figure:
        plt.show()
    else:
        plt.close()

In [ ]:
sequences = data_abundance['cleaned_preds'].tolist()

In [ ]:
mapped_data = visualization.process_protein_contigs_scaffold(
    sequences,          # La tua lista di peptidi (cleaned_preds)
    protein_norm,       # La sequenza della proteina normalizzata
    max_mismatches=1,   # O il valore che stai usando (es. 10)
    min_identity=0.8
)

In [ ]:
mapping_psms_protease_associated_seaborn(
    mapped_sequences=mapped_data,
    prot_seq=protein_norm,
    labels=data_abundance['protease'].tolist(),
    title=f"",
    output_folder=FIGURES_DIR,
    output_file=f"fig2e_{RUN_NAME}_protease_mapping.svg",
    json_colors_path="json/protease_colors.json",
    show_figure=True
)

## Assemblers

In [ ]:
assembler = assembly.Assembler(
    mode="greedy",
    #kmer_size=7,
    min_overlap=3,
    size_threshold=0,
    min_weight=4
)

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_abundance)

In [ ]:
print(MAX_MISMATCHES, MIN_IDENTITY)

In [ ]:
mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, 10, 0.8)

In [ ]:
mapped_scaffolds

In [ ]:
import importlib
importlib.reload(visualization)

In [ ]:
def mapping_substitutions_seaborn(
    mapped_sequences,
    prot_seq,
    bar_colors=None,
    output_folder=".",
    output_file=None,
    contig_color="#fdbb84",
    show_figure=False,
):
    sns.set_style("white")
    sns.set_context("paper", font_scale=1.2)

    default_colors = {
        "match": contig_color,
        "mismatch": "#b30000",
        "D_to_N": "#000000",
        "E_to_Q": "#A8A29E",
    }
    colors = {**default_colors, **(bar_colors or {})}

    common_height = 0.25
    ref_bar_height = common_height
    pep_bar_height = common_height
    track_spacing = 0.30
    base_y_offset = 0.4

    fig, ax = plt.subplots(figsize=(15, 5)) 

    ref_rect = patches.Rectangle(
        (0, 0), len(prot_seq), ref_bar_height,
        linewidth=0, 
        edgecolor='none', 
        facecolor='#e6f0ef',
        zorder=0
    )
    ax.add_patch(ref_rect)

    tracks = {}

    for seq, mapping in tqdm(mapped_sequences, desc="Plotting substitutions"):
        start_index, end_index, mismatches, _ = mapping
        
        placed = False
        sorted_tracks = sorted(tracks.keys())
        
        current_track_num = 0
        
        for track_num in sorted_tracks:
            track_segments = tracks[track_num]
            if not any(max(s, start_index) < min(e, end_index) for s, e in track_segments):
                tracks[track_num].append((start_index, end_index))
                current_track_num = track_num
                placed = True
                break
        
        if not placed:
            current_track_num = len(tracks)
            tracks[current_track_num] = [(start_index, end_index)]

        current_y = base_y_offset + (current_track_num * track_spacing)
        
        base_rect = patches.Rectangle(
            (start_index, current_y), 
            end_index - start_index, 
            pep_bar_height,
            linewidth=0,
            edgecolor='none',
            facecolor=colors["match"],
            alpha=1.0
        )
        ax.add_patch(base_rect)

        for mismatch in mismatches:
            abs_index = start_index + mismatch
            
            if abs_index >= len(prot_seq) or mismatch >= len(seq):
                continue

            ref_aa = prot_seq[abs_index]
            query_aa = seq[mismatch]

            if query_aa == "D" and ref_aa == "N":
                mut_color = colors["D_to_N"]
            elif query_aa == "E" and ref_aa == "Q":
                mut_color = colors["E_to_Q"]
            else:
                mut_color = colors["mismatch"]

            mut_rect = patches.Rectangle(
                (abs_index, current_y), 
                1, 
                pep_bar_height,
                linewidth=0,
                edgecolor='none',
                facecolor=mut_color,
                zorder=10 
            )
            ax.add_patch(mut_rect)

    max_track = len(tracks) if tracks else 0
    ylim_top = base_y_offset + (max_track * track_spacing) + 0.5
    
    ax.set_xlim(0, len(prot_seq))
    ax.set_ylim(0, ylim_top)
    
    ax.set_xlabel("Residue Position", fontsize=14)
    ax.set_yticks([])
    
    sns.despine(left=True, bottom=True)
    ax.spines['bottom'].set_visible(False)
    ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True)

    legend_patches = [
        patches.Patch(color=colors["match"], label="Match"),
        patches.Patch(color=colors["mismatch"], label="Mismatch"),
        patches.Patch(color=colors["D_to_N"], label="Seq:D → Ref:N"),
        patches.Patch(color=colors["E_to_Q"], label="Seq:E → Ref:Q"),
    ]
    
    ax.legend(
        handles=legend_patches, 
        title="",
        loc='upper center', 
        bbox_to_anchor=(0.5, 1.05),
        ncol=4,
        frameon=False,
        fontsize=11
    )

    plt.tight_layout()

    if output_file:
        os.makedirs(output_folder, exist_ok=True)
        full_path = os.path.join(output_folder, output_file)
        plt.savefig(full_path, format='svg', dpi=300, bbox_inches='tight')
        print(f"Figure saved to {full_path}")

    if show_figure:
        plt.show()
    else:
        plt.close()

In [ ]:
mapping_substitutions_seaborn(
    mapped_scaffolds,
    protein_norm,
    output_folder=FIGURES_DIR,
    output_file=f"fig3b_{RUN_NAME}_substitutions_mapping.svg",
    show_figure=True
)

In [ ]:
df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_scaffolds)

In [ ]:
protein_norm